# Projeto - Sprint 10

## Introdução

### Descrição dos Dados

Os dados de exploração geológica para as três regiões estão armazenados em arquivos:

- geo_data_0.csv (https://practicum-content.s3.us-west-1.amazonaws.com/datasets/geo_data_0.csv)
- geo_data_1.csv (https://practicum-content.s3.us-west-1.amazonaws.com/datasets/geo_data_1.csv)
- geo_data_2.csv (https://practicum-content.s3.us-west-1.amazonaws.com/datasets/geo_data_2.csv)
- id — identificador unívoco de poço de petróleo
- f0, f1, f2 — três características de pontos (seu significado específico não é importante, mas as próprias características são significativas)
- product — volume de reservas no poço de petróleo (milhares de barris).

### Condições:

- Apenas regressão linear deve ser usada para o treinamento do modelo.
- Ao explorar a região, um estudo de 500 pontos é realizado e os melhores 200 pontos são selecionados para calcular o lucro.
- O orçamento para o desenvolvimento de 200 poços de petróleo é 100 milhões de dólares.
- Um barril de petróleo bruto traz 4.5 dólares de receita. A receita de uma unidade de produto é 4.500 dólares (o volume de reservas está em milhares de barris).
- Depois de ter avaliado os riscos, mantenha apenas as regiões com o risco de perdas inferior a 2.5%. Entre aquelas que se enquadram no critério, você precisa selecionar a região com o lucro médio mais alto.

Os dados são sintéticos e não incluem nenhum detalhe de contratos ou características de poços.

# 1. Carregando Dados

In [666]:
# Importando bibliotecas
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error


In [667]:
# Carregando dados
geo_data_0 = pd.read_csv("../datasets/geo_data_0.csv")
geo_data_1 = pd.read_csv("../datasets/geo_data_1.csv")
geo_data_2 = pd.read_csv("../datasets/geo_data_2.csv")


## Região 0 (geo_data_0)

In [668]:
geo_data_0.info()
print("Número de observações duplicadas:", geo_data_0.duplicated().sum())
print("Número de ids duplicados:", geo_data_0["id"].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
Número de observações duplicadas: 0
Número de ids duplicados: 10


In [669]:
geo_data_0[geo_data_0["id"].duplicated(keep=False)].sort_values(by="id")

,id,f0,f1,f2,product
66136,74z30,1.084962,-0.312358,6.990771,127.643327
64022,74z30,0.741456,0.459229,5.153109,140.771492
51970,A5aEY,-0.180335,0.935548,-2.094773,33.020205
3389,A5aEY,-0.039949,0.156872,0.209861,89.249364
69163,AGS9W,-0.933795,0.116194,-3.655896,19.230453
42529,AGS9W,1.454747,-0.479651,0.683380,126.370504
931,HZww2,0.755284,0.368511,1.863211,30.681774
7530,HZww2,1.061194,-0.373969,10.430210,158.828695
63593,QcMuo,0.635635,-0.473422,0.862670,64.578675
1949,QcMuo,0.506563,-0.323775,-2.215583,75.496502


Não existem valores nulos no dataset da Região 0, geo_data_0. Não existem observações completamente duplicadas. Embora existam 10 valores duplicados na coluna id, os seus respectivos valores não são idênticos. Como é provável que esses dados representem observações reais e que a duplicação de ids seja apenas um erro na entrada dos dados originais, nenhuma linha sera excluída antes da análise.

## Região 1 (geo_data_1)

In [670]:
geo_data_1.info()
print("Number of duplicated samples:", geo_data_1.duplicated().sum())
print("Number of duplicated ids:", geo_data_1["id"].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
Number of duplicated samples: 0
Number of duplicated ids: 4


In [671]:
geo_data_1[geo_data_1["id"].duplicated(keep=False)].sort_values(by="id")

,id,f0,f1,f2,product
5849,5ltQ6,-3.435401,-12.296043,1.999796,57.085625
84461,5ltQ6,18.213839,2.191999,3.993869,107.813044
1305,LHZR0,11.170835,-1.945066,3.002872,80.859783
41906,LHZR0,-8.989672,-4.286607,2.009139,57.085625
2721,bfPNe,-9.494442,-5.463692,4.006042,110.992147
82178,bfPNe,-6.202799,-4.820045,2.995107,84.038886
47591,wt4Uk,-9.091098,-8.109279,-0.002314,3.179103
82873,wt4Uk,10.259972,-9.376355,4.994297,134.766305


Não existem valores nulos no dataset da Região 1, geo_data_1. Não existem observações completamente duplicadas. Embora existam 4 valores duplicados na coluna id, os seus respectivos valores não são idênticos. Como é provável que esses dados representem observações reais e que a duplicação de ids seja apenas um erro na entrada dos dados originais, nenhuma linha sera excluída antes da análise.

## Região 2 (geo_data_2)

In [672]:
geo_data_2.info()
print("Number of duplicated samples:", geo_data_2.duplicated().sum())
print("Number of duplicated ids:", geo_data_2["id"].duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
Number of duplicated samples: 0
Number of duplicated ids: 4


In [673]:
geo_data_2[geo_data_2["id"].duplicated(keep=False)].sort_values(by="id")

,id,f0,f1,f2,product
45404,KUPhW,0.231846,-1.698941,4.990775,11.716299
55967,KUPhW,1.211150,3.176408,5.543540,132.831802
11449,VF7Jo,2.122656,-0.858275,5.746001,181.716817
49564,VF7Jo,-0.883115,0.560537,0.723601,136.233420
44378,Vcm5J,-1.229484,-2.439204,1.222909,137.968290
95090,Vcm5J,2.587702,1.986875,2.482245,92.327572
28039,xCHr8,1.633027,0.368135,-2.378367,6.120525
43233,xCHr8,-0.847066,2.101796,5.597130,184.388641


Não existem valores nulos no dataset da Região 2, geo_data_2. Não existem observações completamente duplicadas. Embora existam 4 valores duplicados na coluna id, os seus respectivos valores não são idênticos. Como é provável que esses dados representem observações reais e que a duplicação de ids seja apenas um erro na entrada dos dados originais, nenhuma linha sera excluída antes da análise.

# 2. Treinando e Testando um modelo LinearRegression da Scikit-Learn para cada Região

## Ferramentas

Primeiro definimos funções para ajudar no treinamento de modelos:
- features_target_split: divide um dataset entre características e objetivo. 
  - Recebe:
    - O dataset
    - A lista de colunas que não serão usadas para treinamento 
    - A coluna objetivo
  - Retorna:
    - Características
    - Objetivos
- model_pipe: cria uma Pipeline com StandardScaler e ColumnTransformer.
  - Recebe:
    - Lista de colunas numéricas para aplicar o StandardScaler
  - Retorna:
    - A pipeline criada
  - Essa função não é necessária para esse projeto pois estamos treinando um modelo de Linear Regression que não depende ou é afetado por Scaling. Aind assim, para praticar e padronizar a função, decidi aplicar tanto um ColumnTransformer quanto um StandardScaler e demonstrar a funcionalidade de ambos em um projeto real.
- region_worflow: aplica nosso workflow:
  - Chama as funções anteriores (para divisão do dataset e criação de Pipeline)
  - Divide as características e objetivo em datasets de treinamento e validação
  - Treina o modelo
  - Prevê o objetivo no dataset de validação
  - Cria um DataFrame com o objetivo real original, a previsão de objetivo e os respectivos ids, mantendo o índice original do dataset completo original
  - Calcula o Volume Médio do dataset de validação
  - Calcula a Raíz do Erro Quadrático Médio do modelo no dataset de validação
  - Recebe:
    - O dataset da região
    - A lista de colunas que não serão usadas para treinamento 
    - A lista de colunas de características
    - A coluna objetivo
    - A proporção esperada do dataset de validação
    - Um random_state
  - Retorna:
    - DataFrame de objetivos de validação
    - DataFrame de objetivos previstos
    - DataFrame que combina ambos valores de objetivo e os ids originais
    - Volume Médio de barris do dataset de validação
    - A Raíz do Erro Quadrático Médio do modelo no dataset de validação

In [674]:
def features_target_split(data_df:pd.DataFrame,non_training_columns:list[str],target_column:str) -> tuple[pd.DataFrame,pd.Series]:
    features = data_df.drop(non_training_columns + [target_column],axis=1)
    target = data_df.loc[:,target_column]
    return features, target

In [675]:
def model_pipe(num_columns:list[str]):
    scaler = StandardScaler()
    ct = ColumnTransformer(
        transformers=[("num_preprocess", scaler, num_columns)],
        remainder="passthrough"
    )

    pipe = Pipeline([
        ("column_transform", ct),
        ("model", LinearRegression())
    ])
    return pipe

In [676]:
def region_workflow(data_df:pd.DataFrame,non_training_columns,feature_columns,target_column,test_size:float,random_state:int) -> tuple[pd.Series,pd.Series,pd.DataFrame,float,float,Pipeline]:

    features, target = features_target_split(data_df, non_training_columns,target_column)
    features_train, features_valid, target_train, target_valid = train_test_split(features, target, test_size=test_size,random_state=random_state)

    pipe = model_pipe(feature_columns)
    pipe.fit(features_train,target_train)

    predicted_valid = pd.Series(pipe.predict(features_valid),index=target_valid.index)
    validation_results = pd.DataFrame({"actual_product":target_valid,"predicted_product":predicted_valid}).merge(data_df[non_training_columns],how="left",left_index=True,right_index=True)

    mean_predicted = predicted_valid.mean()
    root_mean_square_error = mean_squared_error(target_valid,predicted_valid) ** (1/2)

    return target_valid, predicted_valid, validation_results, mean_predicted,root_mean_square_error,pipe

In [677]:
# Variáveis para controle das funções


split_state = 12345 # random_state para train_test_split
valid_size = 0.25
non_training_columns = ["id"]
feature_columns = ["f0","f1","f2"]
target_column = "product"


### Região 0 (geo_data_0)

In [678]:
(
    target_valid_0,
    predicted_valid_0,
    validation_results_0,
    mean_predicted_0,
    rmse_0,
    pipe_0,
) = region_workflow(
    geo_data_0,
    non_training_columns,
    feature_columns,
    target_column,
    valid_size,
    split_state,
)
print(f"Volume médio previsto de reservas: {mean_predicted_0:.2f}")
print(f"REQM do modelo: {rmse_0:.2f}")


Volume médio previsto de reservas: 92.59
REQM do modelo: 37.58


### Região 1 (geo_data_1)

In [679]:
(
    target_valid_1,
    predicted_valid_1,
    validation_results_1,
    mean_predicted_1,
    rmse_1,
    pipe_1,
) = region_workflow(
    geo_data_1,
    non_training_columns,
    feature_columns,
    target_column,
    valid_size,
    split_state,
)
print(f"Volume médio previsto de reservas: {mean_predicted_1:.2f}")
print(f"REQM do modelo: {rmse_1:.2f}")


Volume médio previsto de reservas: 68.73
REQM do modelo: 0.89


### Região 2 (geo_data_2)

In [680]:
(
    target_valid_2,
    predicted_valid_2,
    validation_results_2,
    mean_predicted_2,
    rmse_2,
    pipe_2,
) = region_workflow(
    geo_data_2,
    non_training_columns,
    feature_columns,
    target_column,
    valid_size,
    split_state,
)
print(f"Volume médio previsto de reservas: {mean_predicted_2:.2f}")
print(f"REQM do modelo: {rmse_2:.2f}")


Volume médio previsto de reservas: 94.97
REQM do modelo: 40.03


##  Análise

- Região 0 tem volume médio previsto de reservas de 92.59 milhares de barris de petróleo com REQM 37.58, o que indica que as predições são incertas.

- Região 1 tem volume médio previsto de reservas de 68.73 milhares de barris de petróleo com REQM 0.89, o que indica que as predições tem alto nível de confiabilidade.

- Região 2 tem volume médio previsto de reservas de 94.97 milhares de barris de petróleo com REQM 40.03, o que indica que as predições são incertas.

Nenhuma das três regiões pode ser escolhida como a melhor região para desenvolvimento usando somente essas informações. Mas podemos afirmar que, aplicando a metodologia de treinamento desses modelos, apenas o modelo da região 1 apresentou incerteza de previsão pequena para futuros poços (assumindo que as características região não mudem e que os dados usados para treinamento são representativos da região). 

Nos baseando nesses valores, e colocando o REQM de lado por um momento, ficaria claro que nenhuma das regiões tem média de volume que atinge ou ultrapassa nossa média mínima de volume de 111,11 milhares de barris de petróleo. Isso significa que, assumindo veracidade desses resultados, nenhuma região pode ser escolhida aleatoriamente para desenvolvimento de poços de petróleo sem que o risco de prejuízo seja totalmente incerto e possivelmente alto.

Mas somente o modelo da região 1 tem confiabilidade o suficiente para fazer boas escolhas de pontos para desenvolvimento. Em outras palavras, se esses dados fossem de aplicações reais desses modelos a dados de futuros pontos para desenvolvimento, os modelos da região 0 e 2 não teriam confiabilidade suficiente nem para excluir as regiões com base na média de volume de cada poço.

# 3. Preparando o Cálculo do Lucro

In [681]:
# Número de poços de Petróleo para desenvolvimento
oil_well_num = 200

# Orçamento de 1 milhões de dólares ($1.000.000) para o desenvolvimento
budget = 100_000_000

# Receita de uma unidade de produto (que equivalem a 1.000 barris de petróleo).
product_unit_rev = 4500

# Risco de perdas máximo em porcentagem. Manter apenas as regiões com risco inferior.
loss_risk_max = 0.025


In [682]:
# Dado o investimento de 100 milhões para 200 poços de petróleo, cada um precisa
# produzir, em média, uma quantidade de unidades equivalente a pelo menos 500 mil
# dólares para evitar prejuízos (isso é aproximadamente 111,1 unidades).
product_mean_min = budget / oil_well_num / product_unit_rev
print(product_mean_min)

111.11111111111111


In [683]:
print(f"Volume médio mínimo de barris para evitar prejuízos: {product_mean_min:.2f}")
product_mean_0 =geo_data_0["product"].mean()
product_mean_1 =geo_data_1["product"].mean()
product_mean_2 =geo_data_2["product"].mean()
print(f"Volume médio real de barris da Região 0: {product_mean_0:.2f}")
print(f"Volume médio real de barris da Região 1: {product_mean_1:.2f}")
print(f"Volume médio real de barris da Região 2: {product_mean_2:.2f}")

Volume médio mínimo de barris para evitar prejuízos: 111.11
Volume médio real de barris da Região 0: 92.50
Volume médio real de barris da Região 1: 68.83
Volume médio real de barris da Região 2: 95.00


Nenhuma das três regiões apresenta um volume médio real de barris de petróleo que supere o volume médio mínimo para evitarmos prejuízos. Isso não significa que nenhuma região tenha potencial de gerar lucros. Nossa tarefa é ser seletivos ao escolher os pontos para possível desenvolvimento de poços de petróleo. Nenhuma das regiões se mostrou ideal em média de volume de barris de petróleo, tanto em dados reais quanto nos 

# 4. Calculando o Lucro dos Volumes Reais dos Top 200 poços com volumes previstos mais altos para cada região

Definindo o problema:
- Para cada região:
  - Top 200 poços em volumes previstos pelos modelos
  - Desses 200 poços, somar o volume __real__ com base nos dados verdadeiros
  - Calcular a receita com base no valor acima
  - Calcular o lucro potencial

### Ferramentas

Função para selecionar os top 200 poços de petróleo, somar a receita com base no volume verdadeiro e calcular o lucro.
Retorna um DataFrame com os top 200 poços, a receita potencial e o lucro potencial calculado

In [684]:
def region_predicted_revenue_profit(region_data:pd.DataFrame,sort_column:str,target_column:str,count:int, budget: int) -> tuple[pd.DataFrame,float]:
    top_wells = region_data.sort_values(by=sort_column,ascending=False).head(count)
    target_product_sum = top_wells[target_column].sum()
    potential_revenue = target_product_sum * product_unit_rev
    profit = potential_revenue - budget
    return top_wells,potential_revenue, profit



In [685]:
top_wells_0, potential_revenue_0, profit_0 = region_predicted_revenue_profit(
    validation_results_0, "predicted_product","actual_product", oil_well_num, budget
)
top_wells_1, potential_revenue_1, profit_1 = region_predicted_revenue_profit(
    validation_results_1, "predicted_product","actual_product", oil_well_num, budget
)
top_wells_2, potential_revenue_2, profit_2 = region_predicted_revenue_profit(
    validation_results_2, "predicted_product","actual_product", oil_well_num, budget
)

print(f"Receita potencial da Região 0 em dólares: {potential_revenue_0:,.2f}")
print(f"Receita potencial da Região 1 em dólares: {potential_revenue_1:,.2f}")
print(f"Receita potencial da Região 2 em dólares: {potential_revenue_2:,.2f}")
print()
print(f"Lucro potencial da Região 0 em dólares: {profit_0:,.2f}")
print(f"Lucro  potencial da Região 1 em dólares: {profit_1:,.2f}")
print(f"Lucro potencial da Região 2 em dólares: {profit_2:,.2f}")

Receita potencial da Região 0 em dólares: 133,208,260.43
Receita potencial da Região 1 em dólares: 124,150,866.97
Receita potencial da Região 2 em dólares: 127,103,499.64

Lucro potencial da Região 0 em dólares: 33,208,260.43
Lucro  potencial da Região 1 em dólares: 24,150,866.97
Lucro potencial da Região 2 em dólares: 27,103,499.64


A Região 0 é a região com o maior valor de lucro potencial se escolhermos exatamente os pontos que foram escolhidos pelo nosso modelo. Isso significa que, se esses pontos forem escolhidos para construção, a Região 0 é a melhor região para construção de 200 poços de petróleo, com um potencial de lucro de 33.2 milhões de dólares. É importante observar que, esses resultados tiveram base no modelo que escolheu esses 200 poços de petróleo. Esse modelo tem um REQM muito alto. Nesse caso específico, a Região 0 apresentou o maior lucro possível. Ainda assim, não quer dizer que uma futura aplicação do mesmo modelo em dados futuros tenha resultados tão positivos quanto esses.

# 5. Calculando Lucro Potencial, Intervalo de Confiança e Risco utilizando técnica Bootstrapping

In [686]:
# Intervalo de Confiança de 95% para a Receita Média (2.5% para cada cauda da
# distribuição)
confidence_interval = (0.025,0.975)

bootstrap_sample_size = 500
bootstrap_samples = 1000

## Ferramentas

Função para calcular lucro, intervalo de confiança e risco

In [687]:
def profit_distribution(region_data:pd.DataFrame,sort_column:str,target_column:str,bootstrap_samples:int,bootstrap_sample_size:int,top_well_num:int, confidence_interval:tuple[float,float]):

    bootstrap_state = np.random.RandomState(12345)
    profits = []

    for i in range(bootstrap_samples):
        subsample = region_data.sample(n=bootstrap_sample_size, replace=True,random_state=bootstrap_state)
        sorted_subsample = subsample.sort_values(by=sort_column,ascending=False)
        top_subsample = sorted_subsample.head(top_well_num)
        total_revenue = top_subsample[target_column].sum() * product_unit_rev
        profit = total_revenue - budget
        profits.append(profit)


    profits = pd.Series(profits)
    lower = profits.quantile(confidence_interval[0])
    upper = profits.quantile(confidence_interval[1])
    profit_mean = profits.mean()
    loss_proba = (profits < 0).mean()

    return profit_mean, lower, upper, loss_proba, profits


### Região 0 (geo_data_0)

In [688]:
profit_mean_0, lower_0, upper_0, loss_proba_0, profits_0 = profit_distribution(validation_results_0,"predicted_product", "actual_product", bootstrap_samples, bootstrap_sample_size, oil_well_num, confidence_interval)

print(f"Média de Lucro para Região 0 (usando técnica bootstrapping): {profit_mean_0:,.2f}")
print(f"Quantil 2.5%: {lower_0:,.2f}")
print(f"Quantil 97.5%: {upper_0:,.2f}")
print(f"Probabilidade de Prejuízos: {loss_proba_0:,.2%}")



Média de Lucro para Região 0 (usando técnica bootstrapping): 3,961,649.85
Quantil 2.5%: -1,112,155.46
Quantil 97.5%: 9,097,669.42
Probabilidade de Prejuízos: 6.90%


### Região 1 (geo_data_1)

In [689]:
profit_mean_1, lower_1, upper_1, loss_proba_1, profits_1 = profit_distribution(validation_results_1,"predicted_product", "actual_product", bootstrap_samples, bootstrap_sample_size, oil_well_num, confidence_interval)

print(f"Média de Lucro para Região 1 (usando técnica bootstrapping): {profit_mean_1:,.2f}")
print(f"Quantil 2.5%: {lower_1:,.2f}")
print(f"Quantil 97.5%: {upper_1:,.2f}")
print(f"Probabilidade de Prejuízos: {loss_proba_1:,.2%}")


Média de Lucro para Região 1 (usando técnica bootstrapping): 4,560,451.06
Quantil 2.5%: 338,205.09
Quantil 97.5%: 8,522,894.54
Probabilidade de Prejuízos: 1.50%


### Região 2 (geo_data_2)

In [690]:
profit_mean_2, lower_2, upper_2, loss_proba_2, profits_2 = profit_distribution(validation_results_2,"predicted_product", "actual_product", bootstrap_samples, bootstrap_sample_size, oil_well_num, confidence_interval)

print(f"Média de Lucro para Região 2 (usando técnica bootstrapping): {profit_mean_2:,.2f}")
print(f"Quantil 2.5%: {lower_2:,.2f}")
print(f"Quantil 97.5%: {upper_2:,.2f}")
print(f"Probabilidade de Prejuízos: {loss_proba_2:,.2%}")


Média de Lucro para Região 2 (usando técnica bootstrapping): 4,044,038.67
Quantil 2.5%: -1,633,504.13
Quantil 97.5%: 9,503,595.75
Probabilidade de Prejuízos: 7.60%


## Conclusão

A única região com risco de prejuízo menor que 2.5% é a Região 1 e essa é a região que deve ser escolhida para desenvolvimento dos 200 poços com base nos nossos modelos de trainmento. Essa é também a região com a maior média de lucro e menor variabilidade no lucro possível. Esse resultado mostra que se aplicarmos nosso modelo da região 1 a futuros dados dessa região, teremos menos probabilidade de termos prejuízos. Essa é uma consequência da alta confiabilidade do modelo da região 1. 

Na questão #4, nós selecionamos os 200 melhores poços em valores previstos pelos modelos. Nessa análise inicial, o resultado foi um lucro alto na Região 0. Na questão #5, nós fomos mais a fundo. Ao invés de selecionar os melhores poços, nós fizemos agrupamentos de 500 poços de todos aqueles que tiveram seus valores previstos pelo modelo, com repetições, e só então selecionamos os 200 melhores __com base nos valores previstos__. Fizemos isso 1.000 vezes e calculamos a distribuição dos lucros. O resultado é uma aproximação da variabilidade dos possíveis lucros resultantes da aplicação dos nossos modelos. Como resultado, obtivemos uma distribuição dos possívels lucros que teríamos se agrupassemos 1.000 diferentes seleções de 500 poços e escolhessemos os 200 melhores usando nossos modelos. Com isso, conseguimos calcular a probabilidade de termos prejuízos em cada uma das regiões se usarmos os modelos treinados para tomar decisões. O resultado é tanto um sinal de que a performace de um modelo deve ser muito bem apresentada e investigada ao tomar decisões baseadas nos resultados desses modelos.